# GPT direct generation vs answer-shape structured reasoning (v2)

This notebook runs a paired experiment on the 160-question BioASQ dev split. Both arms receive exactly the same question and PubMed resources, with no gold answer.

- **Direct baseline:** one extractive answer, `[BE] answer [EE]`.
- **Structured reasoning:** answer shape → requested type → relation and constraints → candidate spans → boundary checks → minimal selected span.

Both arms use `gpt-4.1-mini-2025-04-14`, temperature 0, and official BioASQ Java scoring. Run the fresh 10-question pilot (questions 11–20) first and inspect `reasoning_review.md`. A full run requires approximately 320 successful calls (two per question); cached pilot calls are reused.

This measures whether inference-time structured reasoning helps GPT on this dataset. It does not yet show that Qwen-0.5B or Qwen-3B can learn the procedure.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents, start / 'Task-Structured Counterfactual Preference Mining']:
        if (candidate / 'cse_dpo').is_dir() and (candidate / 'src').is_dir():
            return candidate.resolve()
    raise RuntimeError('Could not locate the project root')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SCRIPT = PROJECT_ROOT / 'cse_dpo/compare_gpt_direct_vs_structured_reasoning_dev.py'
OUTPUT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/inference_strategy_comparisons/gpt41mini_direct_vs_answer_shape_reasoning_dev160_v2'
PILOT_OFFSET = 10  # Fresh questions 11–20; the first 10 informed prompt development.
PILOT_QUESTIONS = 10
RUN_PILOT = False   # Change to True when ready: about 20 successful API calls.
RUN_FULL = False    # Enable only after reviewing the pilot: about 320 total calls.
print('Project:', PROJECT_ROOT)
print('Script:', SCRIPT)
print('Output:', OUTPUT_ROOT)

## 1. Inspect the prompts without making API calls

In [ ]:
dry_run = [
    sys.executable, str(SCRIPT),
    '--question-limit', '1',
    '--dry-run',
]
subprocess.run(dry_run, cwd=PROJECT_ROOT, check=True)

## 2. Run a fresh 10-question v2 pilot

Set `RUN_PILOT=True` in the configuration cell. The script is resumable and caches each arm separately. The first ten dev questions are skipped because they were used to diagnose and design the answer-shape prompt.

In [ ]:
pilot_command = [
    sys.executable, str(SCRIPT),
    '--output-root', str(OUTPUT_ROOT),
    '--reasoning-prompt-version', 'v2',
    '--question-offset', str(PILOT_OFFSET),
    '--question-limit', str(PILOT_QUESTIONS),
    '--progress-every', '1',
]
print(' '.join(map(str, pilot_command)))
if RUN_PILOT:
    subprocess.run(pilot_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Pilot not started. Set RUN_PILOT=True to run it.')

## 3. Review pilot metrics and reasoning traces

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

summary_path = OUTPUT_ROOT / 'summary.json'
comparison_path = OUTPUT_ROOT / 'per_question_comparison.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    display(pd.DataFrame([
        {'method': 'direct', **summary['direct']},
        {'method': 'structured_reasoning', **summary['structured_reasoning']},
    ]))
    display(pd.DataFrame([summary['structured_minus_direct']]))
    print('Transitions:', summary['transition_counts'])
    print('Paired MRR bootstrap:', summary['paired_mrr_bootstrap'])
else:
    print('No completed pilot summary yet.')

if comparison_path.exists():
    rows = json.loads(comparison_path.read_text(encoding='utf-8'))
    review = pd.DataFrame([{
        'question_id': r['question_id'],
        'question': r['question'],
        'direct': r['direct_prediction'],
        'reasoned': r['structured_reasoning_prediction'],
        'gold': r['gold_output'],
        'answer_shape': r['structured_trace'].get('answer_shape'),
        'transition': r['transition'],
        'direct_mrr': r['direct_mrr'],
        'reasoning_mrr': r['structured_reasoning_mrr'],
        'direct_words': r.get('direct_answer_word_count'),
        'reasoning_words': r.get('structured_answer_word_count'),
        'gold_min_words': r.get('gold_min_answer_word_count'),
        'selected_evidence_reason': r['structured_trace']['selection_reason'],
    } for r in rows])
    display(review.sort_values(['transition', 'question_id']))

The detailed human-readable trace audit is written to:

`Artifacts/cse_dpo/inference_strategy_comparisons/gpt41mini_direct_vs_answer_shape_reasoning_dev160_v2/reasoning_review.md`

Check whether the model identifies the correct requested type and relation, considers realistic candidates, and cites evidence that actually supports the selected span.

## 4. Extend the cached pilot to all 160 questions

Only do this if the pilot traces follow the intended process. Set `RUN_FULL=True`. Existing pilot responses will be loaded from cache.

In [ ]:
full_command = [
    sys.executable, str(SCRIPT),
    '--output-root', str(OUTPUT_ROOT),
    '--reasoning-prompt-version', 'v2',
    '--question-limit', '0',
    '--progress-every', '5',
]
print(' '.join(map(str, full_command)))
if RUN_FULL:
    subprocess.run(full_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Full run not started. Set RUN_FULL=True after reviewing the pilot.')

## Interpretation rule

Structured reasoning is promising only if it produces a positive paired MRR change and gains more questions than it loses. On all 160 questions, also inspect the 95% paired bootstrap interval. If it crosses zero, treat the result as inconclusive. Even a positive GPT result establishes a reasoning-process hypothesis, not evidence that rationale SFT or DPO will transfer it to Qwen.